In [1]:
import os, sys
import glob
import pickle
import numpy as np
import pandas as pd
import cv2
from shutil import copyfile
# from scipy.io import wavfile
# from tqdm import tqdm
import torch
# import moviepy.editor as mp
# from facenet_pytorch import MTCNN
from deepface import DeepFace

2023-10-27 15:39:16.266902: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-27 15:39:16.967840: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## db_name = ['Monica', 'Janice', 'Tag', 'Chandler', 'The Interviewer', 'Mr. Treeger', 'Carol',\
##            'Charlie', 'Phoebe', 'Pete', 'Ross', 'Mona', 'Doug', 'Sergei', 'Emily', 'Joey', 'Rachel']

In [2]:
df_train = pd.read_csv('/home/matt/Model/Data/MELD/train/train_sent_emo.csv')
df_test = pd.read_csv('/home/matt/Model/Data/MELD/test_sent_emo.csv')
df_dev = pd.read_csv('/home/matt/Model/Data/MELD/dev_sent_emo.csv')

In [3]:
df_train

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731"
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442"
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389"
3,4,So lets talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572"
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917"
...,...,...,...,...,...,...,...,...,...,...,...
9984,10474,You or me?,Chandler,neutral,neutral,1038,13,2,3,"00:00:48,173","00:00:50,799"
9985,10475,"I got it. Uh, Joey, women don't have Adam's ap...",Ross,neutral,neutral,1038,14,2,3,"00:00:51,009","00:00:53,594"
9986,10476,"You guys are messing with me, right?",Joey,surprise,positive,1038,15,2,3,"00:01:00,518","00:01:03,520"
9987,10477,Yeah.,All,neutral,neutral,1038,16,2,3,"00:01:05,398","00:01:07,274"


In [4]:
df_train['Speaker'] = df_train['Speaker'].astype(str)
df_test['Speaker'] = df_test['Speaker'].astype(str)
df_dev['Speaker'] = df_dev['Speaker'].astype(str)

In [5]:
%%time
%%capture
output_path = r'/home/matt/Model/Database_processed/MELD_RAW_PROCESSED_Face_Verify'
if not os.path.exists(output_path):
    os.makedirs(output_path)
# co = 0
# corrupt = False
for base_path in glob.glob(r'/home/matt/Model/Database_processed/MELD_RAW_PROCESSED_Face/*'):
    # print(base_path)
    for root, dirs, files in os.walk(base_path, topdown=True):
        folder = base_path.split('/')[-1]
        # print(folder)
# split folder name(train_dia700_utt4) to get dia and utt id
        dia, utt = int(folder.split('_')[1][3:]), int(folder.split('_')[2][3:])
        # print(dia, utt)
# get speaker name
        try:
            if folder.split('_')[0] == 'train':
                speaker = df_train.loc[(df_train['Dialogue_ID'] == dia) & (df_train['Utterance_ID'] == utt)]['Speaker'].values[0]
            elif folder.split('_')[0] == 'test':
                speaker = df_test.loc[(df_test['Dialogue_ID'] == dia) & (df_test['Utterance_ID'] == utt)]['Speaker'].values[0]
            else: # folder.split('_')[0] == 'dev'
                speaker = df_dev.loc[(df_dev['Dialogue_ID'] == dia) & (df_dev['Utterance_ID'] == utt)]['Speaker'].values[0]
        except:
            continue
        # print(speaker)
        out_path = os.path.join(output_path, folder)
        if not os.path.exists(out_path):
            # if co == 400:
            #     corrupt = True
            #     break
            # else:
            #     co += 1
            os.makedirs(out_path)
        else:
            continue
        files.sort()
        # print(os.path.join(base_path, files[0]))
        copyfile(os.path.join(base_path, files[0]), os.path.join(out_path,'audio.wav'))
        files = files[1:]
# record the image info in the folder(how many img and how many faces)
        record = {}
        for i in files:
            j, k = i.split('_')[1], (i.split('_')[2]).split('.')[0]
            if int(j) in record.keys():
                record[int(j)].append(int(k))
            else:
                record[int(j)] = [int(k)]
        # print(files)
        db_name = ['Monica', 'Janice', 'Tag', 'Chandler', 'The Interviewer', 'Mr. Treeger', 'Carol',\
            'Charlie', 'Phoebe', 'Pete', 'Ross', 'Mona', 'Doug', 'Sergei', 'Emily', 'Joey', 'Rachel']
        if speaker in db_name:
            for i in record.keys():
                record[i] = sorted(record[i])
                find_face = False
                for j in record[i]:
                    try:
                        file = 'image_' + str(i) + '_' + str(j) + '.jpg'
                        # print(os.path.join(base_path, file))
                        dfs = DeepFace.find(img_path=os.path.join(base_path, file), db_path="/home/matt/Model/Data/MELD/faces_db", detector_backend='mtcnn', model_name='ArcFace', silent=True)
                        # print('ok...')
                        for id in dfs[0]['identity']:
                            name = id.split('/')[-1][:-4]
                            if speaker in name:
                                find_face = True
                                # print('find face...')
                            break
                    except:
                        print('Do not find face..')
                    if find_face:
                        img = 'image_' + str(i) + '.jpg'
                        copyfile(os.path.join(base_path, file), os.path.join(out_path, img))
                        break
                if not find_face:
                    file = 'image_' + str(i) + '_0.jpg'
                    img = 'image_' + str(i) + '.jpg'
                    copyfile(os.path.join(base_path, file), os.path.join(out_path, img))
        else:
            print('not in face db..')
            for i in record.keys():
                file = 'image_' + str(i) + '_0.jpg'
                img = 'image_' + str(i) + '.jpg'
                copyfile(os.path.join(base_path, file), os.path.join(out_path, img))
    # if corrupt:
    #     break
    # break

2023-10-27 15:39:26.506560: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-27 15:39:26.595657: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-27 15:39:26.595806: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

CPU times: user 1h 35min 42s, sys: 3min 31s, total: 1h 39min 13s
Wall time: 1h 40min 46s
